<a href="https://colab.research.google.com/github/Nanda-Lopes/AlgoStudies/blob/main/Stanford_Algorithms_Specialization_Course4_W2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Problema do Caixeiro Viajante (TSP) via Programação Dinâmica

O Problema do Caixeiro Viajante (TSP) busca encontrar um circuito simples que visite cada vértice exatamente uma vez e retorne ao vértice de origem, minimizando a distância total percorrida. A distância entre duas cidades $(x_1, y_1)$ e $(x_2, y_2)$ corresponde à distância euclidiana:
$$d = \sqrt{(x_1 - x_2)^2 + (y_1 - y_2)^2}$$

Para resolver instâncias exatas com $n = 25$ cidades, a enumeração por força bruta ($O(n!)$) é intratável. O algoritmo de Programação Dinâmica de **Bellman-Held-Karp** foi implementado com complexidade de tempo $O(n^2 2^n)$:

1. Uma cidade de origem fixa foi selecionada (cidade 0).
2. Subconjuntos de vértices foram indexados via máscaras de bits (*bitmasks*), onde o $k$-ésimo bit indica a presença da cidade $k$ no subconjunto.
3. Seja $A[S, j]$ o custo mínimo de um caminho que inicia na cidade 0, visita todos os vértices do conjunto $S$ exatamente uma vez e encerra na cidade $j \in S$:
   - Caso base: $A[\{0\}, 0] = 0$; para qualquer $j \neq 0$, $A[\{0\}, j] = \infty$.
   - Relação de recorrência: para subconjuntos de tamanho $m = 2, 3, \dots, n$:
$$A[S, j] = \min_{k \in S, \, k \neq j} \left\{ A[S \setminus \{j\}, k] + d(k, j) \right\}$$
4. Para otimização de memória, o cálculo foi executado iterativamente por tamanho de subconjunto $m$, mantendo em memória apenas os subproblemas do nível anterior ($m-1$).
5. Ao concluir o preenchimento para o conjunto total com as $n$ cidades, o custo mínimo do circuito completo foi obtido calculando o retorno à origem:
   $$\min_{j = 1}^{n-1} \{ A[V, j] + d(j, 0) \}$$

In [3]:
import math
import numpy as np
from numba import njit
from itertools import combinations

def read_tsp(filename):
    with open(filename, "r") as f:
        lines = [line.strip() for line in f if line.strip()]
    num_cities = int(lines[0])
    coords = []
    for line in lines[1:]:
        x, y = map(float, line.split())
        coords.append((x, y))
    return num_cities, coords

def compute_distances(coords):
    n = len(coords)
    dist = np.zeros((n, n), dtype=np.float64)
    for i in range(n):
        for j in range(n):
            if i != j:
                dist[i, j] = math.sqrt((coords[i][0] - coords[j][0])**2 + (coords[i][1] - coords[j][1])**2)
    return dist

@njit
def held_karp_numba(n, dist, masks_by_size, sizes_count):
    inf = 1e12
    dp = np.full((1 << (n - 1), n), inf, dtype=np.float64)
    dp[0, 0] = 0.0

    offset = 1
    for m in range(2, n + 1):
        count = sizes_count[m]
        current_masks = masks_by_size[offset : offset + count]
        offset += count
        for mask in current_masks:
            for j in range(1, n):
                if (mask & (1 << (j - 1))) != 0:
                    prev_mask = mask ^ (1 << (j - 1))
                    min_val = inf
                    if prev_mask == 0:
                        min_val = dist[0, j]
                    else:
                        for k in range(1, n):
                            if (prev_mask & (1 << (k - 1))) != 0:
                                val = dp[prev_mask, k] + dist[k, j]
                                if val < min_val:
                                    min_val = val
                    dp[mask, j] = min_val

    final_mask = (1 << (n - 1)) - 1
    min_tour = inf
    for j in range(1, n):
        cost = dp[final_mask, j] + dist[j, 0]
        if cost < min_tour:
            min_tour = cost

    return min_tour

def solve_tsp_fast(filename):
    n, coords = read_tsp(filename)
    dist = compute_distances(coords)

    masks_list = []
    sizes_count = [0] * (n + 1)
    for m in range(2, n + 1):
        combs = list(combinations(range(n - 1), m - 1))
        sizes_count[m] = len(combs)
        for comb in combs:
            mask = 0
            for bit in comb:
                mask |= (1 << bit)
            masks_list.append(mask)

    masks_array = np.array(masks_list, dtype=np.int32)
    sizes_count_array = np.array(sizes_count, dtype=np.int32)

    return held_karp_numba(n, dist, masks_array, sizes_count_array)

min_cost = solve_tsp_fast("tsp.txt")
print("==================================")
print("CUSTO MÍNIMO EXATO:")
print(min_cost)
print("RESPOSTA OFICIAL (ARREDONDADO PARA BAIXO):")
print(math.floor(min_cost))
print("==================================")

CUSTO MÍNIMO EXATO:
26442.730308954757
RESPOSTA OFICIAL (ARREDONDADO PARA BAIXO):
26442
